# NLP Day 2 · 실습 2. 벡터 유사도로 비슷한 문장 찾기

**연계 이론:** 이론 3. 임베딩의 활용 · 코사인 유사도 · 단어 벡터 평균 · 정적 임베딩

## 1. 이번 실습에서 만들 것

단어 벡터의 **코사인 유사도(Cosine Similarity)**를 계산하고, 여러 단어 벡터의 평균으로 문장을 표현한 뒤 가장 관련 있는 FAQ를 검색합니다.

| 구분 | 활동 | 확인할 결과 |
|---|---|---|
| A형 | 설명용 2차원 벡터에서 유사도 계산, 학습된 fastText 벡터로 FAQ 검색 | 0.8과 0.0, FAQ 검색 순위 |
| B형 | `cosine_similarity()` 함수 완성 후 같은 FAQ 6개 검색 | 유사도 내림차순 상위 3개 |

이번 실습의 **FAQ 6개와 질문**은 5일차의 문장 임베딩 검색 실습에서도 같은 비교 대상으로 사용합니다.

## 2. A형 — 함께 구현하기

### 2.1 작은 벡터로 코사인 유사도 계산하기

코사인 유사도는 두 벡터의 **내적을 각 벡터의 길이로 나누어** 계산합니다. 다음 숫자는 원리를 설명하기 위한 **임의의 벡터**로, 학습된 임베딩 결과는 아닙니다.

In [10]:
import re
import numpy as np

# 계산 원리 확인용 2차원 벡터
apple = np.array([1.0, 0.0])
banana = np.array([0.8, 0.6])
car = np.array([0.0, 1.0])


def cosine_similarity_example(a, b):
    # 두 벡터의 내적
    numerator = np.dot(a, b)                  
    # 두 벡터의 크기 곱
    denominator = np.linalg.norm(a) * np.linalg.norm(b)  
    # 영벡터로 인한 0 나눗셈 방지
    return float(numerator / denominator) if denominator > 0 else 0.0

print("사과 ↔ 바나나:", cosine_similarity_example(apple, banana))
print("사과 ↔ 자동차:", cosine_similarity_example(apple, car))

# 유사도 결과 검증
assert np.isclose(cosine_similarity_example(apple, banana), 0.8)
assert np.isclose(cosine_similarity_example(apple, car), 0.0)

사과 ↔ 바나나: 0.8
사과 ↔ 자동차: 0.0


### 2.2 FAQ 데이터 준비하기

문장 전체를 하나의 벡터로 만들기 위해 **토큰별 임베딩 벡터의 평균**을 사용합니다. 아래 6개 FAQ는 다음 실습에서도 재사용할 데이터입니다.

In [11]:
# 유사도 검색 대상 FAQ 6개
faqs = [
    "상품은 언제 받을 수 있나요?",
    "배송 조회는 어디에서 하나요?",
    "구매한 상품을 반품하고 싶어요.",
    "결제가 계속 실패해요.",
    "계정 비밀번호를 바꾸고 싶어요.",
    "주문을 취소하려면 어떻게 하나요?",
]
# 기준 질문
query = "배송이 언제 도착하나요?"


def tokenize_korean(text):
    """설명용: 공백·구두점을 기준으로 단순 토큰화합니다."""
    # 한글·영문·숫자 토큰 추출
    return re.findall(r"[가-힣A-Za-z0-9]+", text.lower())

print("질문 토큰:", tokenize_korean(query))
print("FAQ 개수:", len(faqs))
assert len(faqs) == 6

질문 토큰: ['배송이', '언제', '도착하나요']
FAQ 개수: 6


### 2.3 학습된 fastText 벡터 준비하기

이 실습에서는 **실제로 학습된 단어 벡터**를 사용합니다. 공식 한국어 fastText 벡터에서 필요한 단어만 추출한 파일이 `assets/nlp_fasttext_ko_subset.vec`에 있다면 이를 읽습니다(첫 줄에 `단어수 차원수`가 있는 word2vec 텍스트 형식).

그 파일이 없다면 같은 코드를 실행할 수 있도록 **수업용 소규모 한국어 문장 모음에서 fastText를 직접 학습**합니다. 이 경우는 대규모 사전학습 모델이 아니므로, 검색 순위가 반드시 의미적으로 정확하지는 않습니다. 두 경우를 구분해 결과를 해석합니다.

*필요 라이브러리:* `numpy`, `gensim` (설치되지 않았다면 프로젝트 환경에서 `uv add numpy gensim`으로 추가할 수 있습니다).

In [12]:
from pathlib import Path
from gensim.models import FastText, KeyedVectors

# 수업용 소규모 학습 말뭉치 · 검색 성능 보장 없음
small_corpus = [
    # 배송 예정일
    "상품은 언제 받을 수 있나요", "배송이 언제 도착하나요", "상품 배송이 언제 도착하나요",
    "택배는 언제 받을 수 있나요", "물건이 언제 도착하는지 알고 싶어요", "배송 예정일이 궁금해요",
    "언제 상품을 수령할 수 있나요", "구매한 물건은 언제 도착하나요",
    # 배송 조회
    "배송 조회는 어디에서 하나요", "택배 위치를 조회하고 싶어요", "배송 상태를 확인하고 싶어요",
    "운송장 번호로 배송 조회를 합니다", "택배가 지금 어디에 있나요", "배송 진행 상황을 알고 싶어요",
    # 반품
    "구매한 상품을 반품하고 싶어요", "물건을 돌려보내고 싶어요", "반품 절차는 어떻게 되나요",
    "반품 접수를 하고 싶어요", "상품을 반품하는 방법 알려주세요", "구입한 물건을 반품하고 싶어요",
    # 결제
    "결제가 계속 실패해요", "카드 결제가 안 돼요", "결제 오류가 발생했어요",
    "계속 결제할 수 없어요", "결제 실패 이유가 궁금해요", "결제 수단을 확인하고 싶어요",
    # 계정
    "계정 비밀번호를 바꾸고 싶어요", "비밀번호 변경 방법이 궁금해요", "로그인 암호를 변경하고 싶어요",
    "비밀번호 재설정을 하고 싶어요", "계정 암호를 잊어버렸어요",
    # 취소
    "주문을 취소하려면 어떻게 하나요", "주문 취소를 하고 싶어요", "구매한 주문을 취소하고 싶어요",
    "주문 취소 방법 알려주세요", "구매 취소를 신청하고 싶어요",
]

# 사전학습 단어 벡터 파일 경로
vector_path = Path.cwd() / "assets" / "nlp_fasttext_ko_subset.vec"
if vector_path.exists():
    # 파일 존재 시 사전학습 벡터 로딩
    word_vectors = KeyedVectors.load_word2vec_format(str(vector_path), binary=False)
    vector_source = "준비된 한국어 fastText 사전학습 단어 벡터"
else:
    # 파일 부재 시 수업용 fastText 직접 학습
    training_tokens = [tokenize_korean(s) for s in small_corpus]
    tiny_fasttext = FastText(
        sentences=training_tokens,
        vector_size=64,
        window=3,
        min_count=1,
        sg=0,
        workers=1,
        epochs=80,
        seed=42,
    )
    # 학습된 단어 벡터 추출
    word_vectors = tiny_fasttext.wv
    vector_source = "수업용 소규모 문장에서 직접 학습한 fastText 벡터"

# 사용 중인 벡터 출처·차원 확인
print("사용 중인 벡터:", vector_source)
print("단어 벡터 차원:", word_vectors.vector_size)

사용 중인 벡터: 수업용 소규모 문장에서 직접 학습한 fastText 벡터
단어 벡터 차원: 64


### 2.4 단어 벡터 평균으로 FAQ 검색하기

각 문장의 단어 벡터를 평균 내어 문장 벡터를 얻습니다. 미등록 단어가 있는 경우 이용 가능한 단어 벡터만 평균합니다. 실제 검색 결과는 사용한 벡터와 토큰화 방식에 따라 달라집니다.

In [13]:
def sentence_vector(text, vectors):
    # 토큰별 단어 임베딩 수집
    token_vectors = []
    for token in tokenize_korean(text):
        try:
            token_vectors.append(vectors[token])
        except KeyError:
            # 미등록 토큰 제외
            continue
    if not token_vectors:
        # 사용 가능한 벡터가 없을 때 영벡터 반환
        return np.zeros(vectors.vector_size, dtype=np.float32)
    # 단어 벡터 평균으로 문장 벡터 생성
    return np.mean(np.asarray(token_vectors), axis=0)

# 질문·FAQ 문장 벡터 변환
query_vector = sentence_vector(query, word_vectors)
faq_vectors = [sentence_vector(faq, word_vectors) for faq in faqs]

print("질문 벡터 Shape:", query_vector.shape)
print("FAQ 벡터 개수:", len(faq_vectors))
print("FAQ 벡터 차원:", faq_vectors[0].shape)

# 질문과 FAQ의 코사인 유사도 계산
scores_a = [cosine_similarity_example(query_vector, v) for v in faq_vectors]
# 유사도 내림차순 정렬
ranked_a = sorted(zip(faqs, scores_a), key=lambda x: x[1], reverse=True)
# 상위 3개 FAQ 출력
for rank, (faq, score) in enumerate(ranked_a[:3], start=1):
    print(f"{rank}위: {faq} | 유사도 {score:.4f}")

assert query_vector.shape == (word_vectors.vector_size,)
assert len(faq_vectors) == 6

질문 벡터 Shape: (64,)
FAQ 벡터 개수: 6
FAQ 벡터 차원: (64,)
1위: 상품은 언제 받을 수 있나요? | 유사도 0.5262
2위: 배송 조회는 어디에서 하나요? | 유사도 0.4098
3위: 구매한 상품을 반품하고 싶어요. | 유사도 0.3789


**A형 결과 해석**

- 단어의 임베딩 차원과 평균으로 만든 문장 벡터의 차원은 같습니다.
- 질문과 FAQ를 **동일한 임베딩 방식**으로 표현한 뒤 벡터 유사도를 비교합니다.
- 단어 벡터의 단순 평균은 어순을 직접 반영하지 못합니다. 소규모 학습 벡터의 검색 순위가 기대와 다를 수 있으며, 실제 결과를 기준으로 판단합니다.

## 3. B형 — 직접 완성하기

> **문제:** 내적과 벡터의 길이를 사용해 `cosine_similarity()`를 완성하세요. 질문 **`배송이 언제 도착하나요?`**와 여섯 개 FAQ의 유사도를 구하고 **상위 3개**를 출력하세요.

A형에 있는 `np.dot()`과 `np.linalg.norm()` 계산을 참고하세요. 벡터의 크기가 0이면 나눗셈을 하지 않도록 처리합니다.

In [14]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    # ★ TODO: 내적과 벡터 크기를 이용해 코사인 유사도를 반환하세요.
    # ★ 참고: a와 b 중 길이가 0인 벡터가 있다면 0.0을 반환합니다.
    return None

print("확인용 사과-바나나 유사도:", cosine_similarity(apple, banana))
print("확인용 사과-자동차 유사도:", cosine_similarity(apple, car))

확인용 사과-바나나 유사도: None
확인용 사과-자동차 유사도: None


### 3.1 B형 결과 확인하기

함수 완성 후, 설명용 벡터에서 0.8과 0.0을 확인합니다. 같은 함수를 그대로 사용해 질문 벡터와 FAQ 벡터를 비교합니다.

In [15]:
demo_sim_1 = cosine_similarity(apple, banana)
demo_sim_2 = cosine_similarity(apple, car)

if demo_sim_1 is None or demo_sim_2 is None:
    print("B형 함수의 TODO를 완성하면 FAQ 검색을 실행할 수 있습니다.")
else:
    assert np.isclose(demo_sim_1, 0.8), "사과-바나나 유사도는 0.8입니다."
    assert np.isclose(demo_sim_2, 0.0), "사과-자동차 유사도는 0.0입니다."

    scores_b = [cosine_similarity(query_vector, v) for v in faq_vectors]
    ranked_b = sorted(zip(faqs, scores_b), key=lambda x: x[1], reverse=True)
    print("B형: FAQ 유사도 상위 3개")
    for rank, (faq, score) in enumerate(ranked_b[:3], start=1):
        print(f"{rank}위: {faq} | 유사도 {score:.4f}")

    assert len(ranked_b) == 6
    assert all(ranked_b[i][1] >= ranked_b[i + 1][1] for i in range(5))

B형 함수의 TODO를 완성하면 FAQ 검색을 실행할 수 있습니다.


## 4. 결과 확인과 이번 내용 간단하게 요약

**완료 기준:** 설명용 벡터에서 유사도가 **0.8과 0.0**으로 계산되고, FAQ 6개를 유사도 내림차순으로 정렬한 뒤 상위 3개가 출력됩니다.

검색 결과를 보고 다음을 짧게 설명해 보세요.

- 가장 높은 순위의 FAQ는 질문의 의도와 관련이 있나요?
- 단어 벡터를 평균하는 방식이 문장의 의미를 놓칠 수 있는 사례는 무엇인가요?

**이번 내용 간단하게 요약**

첫째, 코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 비교합니다.

둘째, 단어 벡터를 평균하면 길이가 서로 다른 문장도 같은 차원의 벡터로 표현할 수 있습니다.

셋째, 질문과 FAQ 벡터의 유사도를 계산하면 관련 문장을 검색할 수 있습니다. 다만 실제 검색 결과는 임베딩의 학습 데이터와 단순 평균의 한계에 영향을 받습니다.

이 FAQ 데이터는 이후 **문맥 기반 문장 임베딩 검색**과 비교하는 데 다시 사용합니다.